# Function Calling with GigaChat


## Environment Setup

Make sure `.env` contains:

```env
API_KEY=your_access_key
MODEL_NAME=GigaChat
```

`MODEL_NAME` is optional; default is `GigaChat`.


In [ ]:
# Optional: install minimal dependencies in the active environment

# !pip install gigachat python-dotenv --quiet


In [75]:
import os
import re
import json
from dotenv import load_dotenv

load_dotenv()

GIGACHAT_CREDENTIALS = os.getenv('API_KEY') or os.getenv('GIGACHAT_CREDENTIALS') or ''
MODEL_NAME = os.getenv('MODEL_NAME') or 'GigaChat'

print('MODEL_NAME:', MODEL_NAME)
print('Credentials loaded:', bool(GIGACHAT_CREDENTIALS))


MODEL_NAME: GigaChat
Credentials loaded: True


In [76]:
from gigachat import GigaChat
from gigachat.models import Chat, Messages, MessagesRole, Function, FunctionParameters
from gigachat.models.chat import ChatFunctionCall
from gigachat.models.function_parameters_property import FunctionParametersProperty

if not GIGACHAT_CREDENTIALS:
    raise ValueError('Set API_KEY or GIGACHAT_CREDENTIALS in .env before running model calls.')

client = GigaChat(credentials=GIGACHAT_CREDENTIALS, verify_ssl_certs=False)

def _apply_stop_sequences(text, stop_sequences):
    if not stop_sequences:
        return text
    stop_at = None
    for seq in stop_sequences:
        idx = text.find(seq)
        if idx != -1 and (stop_at is None or idx < stop_at):
            stop_at = idx
    return text if stop_at is None else text[:stop_at]

def _normalize_messages(messages, system_prompt='', prefill=''):
    normalized = []
    if system_prompt:
        normalized.append(Messages(role=MessagesRole.SYSTEM, content=system_prompt))

    for message in messages:
        if isinstance(message, Messages):
            normalized.append(message)
            continue

        message_kwargs = {}
        if isinstance(message, dict):
            role = message.get('role', 'user')
            content = message.get('content', '')
            if 'name' in message:
                message_kwargs['name'] = message['name']
            if 'function_call' in message:
                message_kwargs['function_call'] = message['function_call']
        else:
            role = 'user'
            content = str(message)

        role_map = {
            'system': MessagesRole.SYSTEM,
            'assistant': MessagesRole.ASSISTANT,
            'user': MessagesRole.USER,
            'function': MessagesRole.FUNCTION,
        }
        normalized.append(
            Messages(
                role=role_map.get(role, MessagesRole.USER),
                content=content,
                **message_kwargs,
            )
        )

    if prefill:
        normalized.append(Messages(role=MessagesRole.ASSISTANT, content=prefill))

    return normalized

def get_completion(prompt_or_messages, system_prompt='', prefill='', stop_sequences=None, temperature=0.0):
    if isinstance(prompt_or_messages, (list, tuple)):
        messages = _normalize_messages(prompt_or_messages, system_prompt=system_prompt, prefill=prefill)
    else:
        messages = []
        if system_prompt:
            messages.append(Messages(role=MessagesRole.SYSTEM, content=system_prompt))
        messages.append(Messages(role=MessagesRole.USER, content=str(prompt_or_messages)))
        if prefill:
            messages.append(Messages(role=MessagesRole.ASSISTANT, content=prefill))

    chat = Chat(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=temperature,
        messages=messages,
    )
    response = client.chat(chat)
    text = response.choices[0].message.content
    return _apply_stop_sequences(text, stop_sequences)


def get_completion_native(messages, system_prompt='', functions=None, function_call='auto', temperature=0.0):
    normalized = _normalize_messages(messages, system_prompt=system_prompt)

    chat_kwargs = {
        'model': MODEL_NAME,
        'max_tokens': 2000,
        'temperature': temperature,
        'messages': normalized,
    }
    if functions is not None:
        chat_kwargs['functions'] = functions
    if function_call is not None:
        chat_kwargs['function_call'] = function_call

    chat = Chat(**chat_kwargs)
    return client.chat(chat)


In [77]:
# Quick connectivity check
print(get_completion('Say hello in one short sentence.'))


Привет!


---
## Part 1. Function Calling Fundamentals

Function calling means the model decides **when** to use tools and **which parameters** to pass.

In this notebook we use a text-based function-call format:

```xml
<function_calls>
  <invoke name="tool_name">
    <parameter name="arg1">value</parameter>
  </invoke>
</function_calls>
```

Then Python executes the tool, returns `<function_results>`, and GigaChat produces the final answer.


In [78]:
system_prompt_tools_general = """You have access to a set of functions you can use to answer the user's question.
Use tools only when they are needed for factual retrieval or computation.

If a tool is needed, respond using this structure:
<function_calls>
<invoke name="TOOL_NAME">
<parameter name="PARAM">VALUE</parameter>
</invoke>
</function_calls>

After tool execution, you may receive results in this structure:
<function_results>
<result>
<tool_name>...</tool_name>
<stdout>...</stdout>
</result>
</function_results>

When <function_results> is provided, use it to answer the user directly in plain text.
Do not repeat the same tool call unless required parameters are missing or execution failed.
If no tool is needed, answer normally.
"""


In [79]:
system_prompt_tools_calculator = """Here are available tools:
<tools>
<tool_description>
<tool_name>calculator</tool_name>
<description>Performs basic arithmetic for numeric questions.</description>
<parameters>
<parameter><name>first_operand</name><type>number</type></parameter>
<parameter><name>second_operand</name><type>number</type></parameter>
<parameter><name>operator</name><type>str</type><description>One of +, -, *, /</description></parameter>
</parameters>
</tool_description>
</tools>

Special rule:
For any arithmetic or numeric computation request, you MUST call calculator first.
Do not compute the result mentally.
"""

system_prompt_calculator = system_prompt_tools_general + "\n" + system_prompt_tools_calculator

system_prompt_calculator_native = """You can use the calculator function for arithmetic.
Rules:
1. For any arithmetic or numeric computation request, call calculator first.
2. Use the exact numbers from the user request.
3. After receiving the function result, answer in plain text.
4. Do not output XML tags like <function_calls>.
"""

stop_sequences = ['</function_calls>']


In [80]:
# Step 1: ask for a tool call
message = {'role': 'user', 'content': 'Multiply 1984135 by 9343116.'}
first_response = get_completion([message], system_prompt=system_prompt_calculator, stop_sequences=stop_sequences)
# first_response = get_completion([message], system_prompt=system_prompt_calculator)
print(first_response)


<function_calls>
<invoke name="calculator">
<parameter name="first_operand" value="1984135"/>
<parameter name="second_operand" value="9343116"/>
<parameter name="operator" value="*"/>
</invoke>



In [81]:
def parse_invocations(function_call_text):
    invocations = []

    invoke_pattern = re.compile(
        r'<invoke\b[^>]*\bname\s*=\s*(["\'])(.*?)\1[^>]*>(.*?)</invoke>',
        re.IGNORECASE | re.DOTALL,
    )
    param_pair_pattern = re.compile(
        r'<(?:\w+:)?parameter\b([^>]*)>(.*?)</(?:\w+:)?parameter>',
        re.IGNORECASE | re.DOTALL,
    )
    param_self_pattern = re.compile(
        r'<(?:\w+:)?parameter\b([^>]*)/>',
        re.IGNORECASE | re.DOTALL,
    )
    name_attr_pattern = re.compile(r'\bname\s*=\s*(["\'])(.*?)\1', re.IGNORECASE | re.DOTALL)
    value_attr_pattern = re.compile(r'\bvalue\s*=\s*(["\'])(.*?)\1', re.IGNORECASE | re.DOTALL)

    for _, tool_name, body in invoke_pattern.findall(function_call_text):
        params = {}

        # Standard parameter tags: <parameter name="x">value</parameter>
        for attrs, raw_value in param_pair_pattern.findall(body):
            name_match = name_attr_pattern.search(attrs)
            if not name_match:
                continue
            key = name_match.group(2).strip()
            value = raw_value.strip()
            if not value:
                value_match = value_attr_pattern.search(attrs)
                if value_match:
                    value = value_match.group(2).strip()
            if key:
                params[key] = value

        # Self-closing parameter tags: <parameter name="x" value="y"/>
        for attrs in param_self_pattern.findall(body):
            name_match = name_attr_pattern.search(attrs)
            if not name_match:
                continue
            key = name_match.group(2).strip()
            value_match = value_attr_pattern.search(attrs)
            value = value_match.group(2).strip() if value_match else ''
            if key and key not in params:
                params[key] = value

        # Some model variants may pass arguments in a dedicated JSON block.
        if not params:
            json_arg_match = re.search(
                r'<(?:\w+:)?arguments\b[^>]*>(.*?)</(?:\w+:)?arguments>',
                body,
                re.IGNORECASE | re.DOTALL,
            )
            if json_arg_match:
                raw = json_arg_match.group(1).strip()
                try:
                    parsed = json.loads(raw)
                    if isinstance(parsed, dict):
                        params = {str(k): str(v) for k, v in parsed.items()}
                except json.JSONDecodeError:
                    pass

        # Last-resort fallback: parse first JSON object found in invoke body.
        if not params:
            json_obj_match = re.search(r'\{.*?\}', body, re.DOTALL)
            if json_obj_match:
                raw = json_obj_match.group(0).strip()
                try:
                    parsed = json.loads(raw)
                    if isinstance(parsed, dict):
                        params = {str(k): str(v) for k, v in parsed.items()}
                except json.JSONDecodeError:
                    pass

        invocations.append({'tool_name': tool_name.strip(), 'params': params})

    return invocations

def _parse_number(value):
    if isinstance(value, (int, float)):
        return value
    text = str(value).strip()
    if re.fullmatch(r'[-+]?\d+', text):
        return int(text)
    return float(text)


def calculator(first_operand, second_operand, operator):
    a = _parse_number(first_operand)
    b = _parse_number(second_operand)

    if operator == '+':
        return a + b
    if operator == '-':
        return a - b
    if operator == '*':
        return a * b
    if operator == '/':
        if b == 0:
            return 'Error: division by zero'
        return a / b

    return 'Error: unsupported operator'


def construct_function_results(results):
    chunks = []
    for item in results:
        chunks.append(
            '<result>\n'
            f"<tool_name>{item['tool_name']}</tool_name>\n"
            '<stdout>\n'
            f"{item['tool_result']}\n"
            '</stdout>\n'
            '</result>'
        )
    return '<function_results>\n' + '\n'.join(chunks) + '\n</function_results>'


def _fallback_result_text(results):
    if not results:
        return 'No tool results available.'
    if len(results) == 1:
        return str(results[0]['tool_result'])
    return '\n'.join(f"{r['tool_name']}: {r['tool_result']}" for r in results)


def _recover_calculator_params(user_text, params):
    params = dict(params or {})

    needed = {'first_operand', 'second_operand', 'operator'}
    if needed.issubset(params.keys()):
        return params

    text = str(user_text)
    lower = text.lower()

    operator = params.get('operator')
    if not operator:
        symbol_match = re.search(r'([+\-*/x×])', text)
        if symbol_match:
            op = symbol_match.group(1)
            operator = '*' if op in ('x', '×') else op
        elif any(k in lower for k in ('multiply', 'times', 'product')):
            operator = '*'
        elif any(k in lower for k in ('add', 'plus', 'sum')):
            operator = '+'
        elif any(k in lower for k in ('subtract', 'minus', 'difference')):
            operator = '-'
        elif any(k in lower for k in ('divide', 'quotient')):
            operator = '/'

    # Preserve first mention order from user text.
    numbers = re.findall(r'[-+]?\d+(?:\.\d+)?', text)
    if 'first_operand' not in params and len(numbers) >= 1:
        params['first_operand'] = numbers[0]
    if 'second_operand' not in params and len(numbers) >= 2:
        params['second_operand'] = numbers[1]
    if operator and 'operator' not in params:
        params['operator'] = operator

    return params

def run_tool_loop(user_message, system_prompt, tool_map, max_rounds=3, verbose=False):
    messages = [{'role': 'user', 'content': user_message}]
    last_results = []

    for _ in range(max_rounds):
        assistant_raw = get_completion(messages, system_prompt=system_prompt, stop_sequences=stop_sequences)
        if verbose:
            print('<ASSISTANT>')
            print(assistant_raw)

        if '<function_calls>' not in assistant_raw:
            return assistant_raw

        assistant_with_closer = assistant_raw + '</function_calls>'

        invocations = parse_invocations(assistant_with_closer)



        if not invocations:
            return assistant_raw

        results = []
        for inv in invocations:
            tool_name = inv['tool_name']
            params = inv['params']

            if tool_name == 'calculator':
                params = _recover_calculator_params(user_message, params)

            if tool_name not in tool_map:
                tool_result = f"Error: unknown tool '{tool_name}'"
            else:
                try:
                    tool_result = tool_map[tool_name](**params)
                except Exception as e:
                    tool_result = f"Error while running {tool_name}: {type(e).__name__}: {e}"
            
            if verbose:
                print(f'<{tool_name}>')
                print(f'{tool_result}')
            results.append({'tool_name': tool_name, 'tool_result': tool_result})

        last_results = results
        function_results = construct_function_results(results)

        messages.append({'role': 'assistant', 'content': assistant_with_closer})
        messages.append({'role': 'user', 'content': function_results})

    final_response = get_completion(messages, system_prompt=system_prompt)
    if '<function_calls>' in final_response:
        return _fallback_result_text(last_results)
    return final_response


def build_calculator_function_spec():
    return Function(
        name='calculator',
        description='Performs basic arithmetic for numeric questions.',
        parameters=FunctionParameters(
            type='object',
            properties={
                'first_operand': FunctionParametersProperty(type='number', description='First number.'),
                'second_operand': FunctionParametersProperty(type='number', description='Second number.'),
                'operator': FunctionParametersProperty(
                    type='string',
                    description='One of +, -, *, /',
                    enum=['+', '-', '*', '/'],
                ),
            },
            required=['first_operand', 'second_operand', 'operator'],
        ),
    )


calculator_function_specs = [build_calculator_function_spec()]


def _looks_like_arithmetic_query(text):
    text = str(text)
    lower = text.lower()
    keywords = ('multiply', 'times', 'product', 'add', 'sum', 'subtract', 'minus', 'divide', 'calculate')
    if any(k in lower for k in keywords):
        return True
    return bool(re.search(r'\d\s*[-+*/x×]\s*\d', text))


def _coerce_function_args(raw_args):
    if raw_args is None:
        return {}
    if isinstance(raw_args, dict):
        return raw_args
    if isinstance(raw_args, str):
        raw_args = raw_args.strip()
        if not raw_args:
            return {}
        try:
            parsed = json.loads(raw_args)
        except json.JSONDecodeError:
            return {}
        return parsed if isinstance(parsed, dict) else {}
    return {}


def run_tool_loop_native(
    user_message,
    system_prompt,
    tool_map,
    function_specs,
    max_rounds=4,
    force_arithmetic_tool=True,
    verbose=False
):
    messages = [{'role': 'user', 'content': user_message}]
    last_results = []

    for round_idx in range(max_rounds):
        force_calculator_now = (
            force_arithmetic_tool
            and round_idx == 0
            and 'calculator' in tool_map
            and _looks_like_arithmetic_query(user_message)
        )
        function_call_policy = ChatFunctionCall(name='calculator') if force_calculator_now else 'auto'

        response = get_completion_native(
            messages,
            system_prompt=system_prompt,
            functions=function_specs,
            function_call=function_call_policy,
            temperature=0.0,
        )
        choice = response.choices[0]
        assistant_message = choice.message
        messages.append(assistant_message)
        if verbose:
            print(f'<ASSISTANT>:\n{assistant_message}')

        function_call = assistant_message.function_call
        assistant_text = (assistant_message.content or '').strip()
        if verbose:
            print(f'<ASSISTANT>:\n{assistant_text}')

        # Compatibility fallback: some models may still emit XML-style tool calls in text mode.
        if function_call is None and '<function_calls>' in assistant_text:
            xml_text = assistant_text if '</function_calls>' in assistant_text else assistant_text + '</function_calls>'
            parsed = parse_invocations(xml_text)
            if parsed:
                function_call = type('TmpCall', (), {
                    'name': parsed[0]['tool_name'],
                    'arguments': parsed[0]['params'],
                })()

        if function_call is None:
            if assistant_text:
                return assistant_text
            return _fallback_result_text(last_results)

        tool_name = function_call.name
        params = _coerce_function_args(function_call.arguments)

        if tool_name == 'calculator':
            params = _recover_calculator_params(user_message, params)

        if tool_name not in tool_map:
            tool_result = f"Error: unknown tool '{tool_name}'"
        else:
            try:
                tool_result = tool_map[tool_name](**params)
            except Exception as e:
                tool_result = f"Error while running {tool_name}: {type(e).__name__}: {e}"

        last_results = [{'tool_name': tool_name, 'tool_result': tool_result}]
        if verbose:
            print(f'<{tool_name}>:\n{tool_result}')


        if force_arithmetic_tool and tool_name == 'calculator' and _looks_like_arithmetic_query(user_message):
            return (
                f"The result of {params.get('first_operand')} {params.get('operator')} {params.get('second_operand')} "
                f"is {tool_result}, computed using the calculator tool."
            )

        messages.append({'role': 'function', 'name': tool_name, 'content': str(tool_result)})

    response = get_completion_native(
        messages,
        system_prompt=system_prompt,
        functions=function_specs,
        function_call='none',
        temperature=0.0,
    )
    final_text = (response.choices[0].message.content or '').strip()
    if final_text:
        return final_text
    return _fallback_result_text(last_results)


In [82]:
final_answer = run_tool_loop(
    user_message='Multiply 1984135 by 9343116, then explain the result in one sentence. USE CALCULATOR TOOL!!',
    # user_message='Multiply 1984135 by 9343116, then explain the result in one sentence',
    system_prompt=system_prompt_calculator,
    tool_map={'calculator': calculator},
    verbose=True,
)
print(final_answer)


<ASSISTANT>
<function_calls>
<invoke name="calculator">
<parameter name="first_operand" value="1984135"/>
<parameter name="second_operand" value="9343116"/>
<parameter name="operator" value="*"/>
</invoke>

<calculator>
18538003464660
<ASSISTANT>
The product of 1984135 and 9343116 is 18,538,003,464,660. This large number represents the total when multiplying two significant integers.
The product of 1984135 and 9343116 is 18,538,003,464,660. This large number represents the total when multiplying two significant integers.


In [55]:
final_answer = run_tool_loop_native(
    user_message='Multiply 1984135 by 9343116, then explain the result in one sentence.',
    system_prompt=system_prompt_calculator_native,
    tool_map={'calculator': calculator},
    function_specs=calculator_function_specs,
    verbose=True
)
print(final_answer)


<ASSISTANT>:
role='assistant' content='' function_call=FunctionCall(name='calculator', arguments={'first_operand': 1984135, 'operator': '*', 'second_operand': 9343116}) name=None attachments=None data_for_context=None functions_state_id='019c8e85-f6ce-77f2-abcb-9af10999873a' reasoning_content=None id_=None
<ASSISTANT>:

<calculator>:
18538003464660
The result of 1984135 * 9343116 is 18538003464660, computed using the calculator tool.


### Exercise 1 (10-12 min): Trigger the Right Tool

Edit only `USER_PROMPT` so GigaChat calls `calculator`.

Success criterion: output contains `<invoke name="calculator">`.


In [9]:
USER_PROMPT = '[Replace this text]'

response = get_completion(
    [{'role': 'user', 'content': USER_PROMPT}],
    system_prompt=system_prompt_calculator,
    stop_sequences=stop_sequences,
)

is_correct = '<invoke name="calculator">' in response
print(response)
print('\nSolved:', is_correct)


The task asks for replacing some text. While there isn't a specific tool mentioned for this purpose, we can simply replace the given text with whatever new content is desired.

New content: "Hello, world!"

Final Answer: Hello, world!

Solved: False


### Exercise 2 (10-12 min): Avoid Unnecessary Tool Calls

Edit only `USER_PROMPT` so the model **does not** call a tool and answers directly.

Success criterion: output does not contain `<function_calls>`.


In [10]:
USER_PROMPT = '[Replace this text]'

response = get_completion(
    [{'role': 'user', 'content': USER_PROMPT}],
    system_prompt=system_prompt_calculator,
    stop_sequences=stop_sequences,
)

is_correct = '<function_calls>' not in response
print(response)
print('\nSolved:', is_correct)


The instruction has been replaced. Please provide a new question or task.

Solved: True


---
## Part 2. Multi-Tool Low-Code Assistant

Now we use several simple tools that simulate a student support desk.
Students mostly edit natural-language prompts and tool descriptions.


In [56]:
# Small in-memory data for tools
campus_hours_data = {
    'monday': '08:00-20:00',
    'tuesday': '08:00-20:00',
    'wednesday': '08:00-20:00',
    'thursday': '08:00-20:00',
    'friday': '08:00-18:00',
    'saturday': '10:00-16:00',
    'sunday': 'closed',
}

room_capacity_data = {
    'A101': 24,
    'B202': 40,
    'C303': 12,
}

deadline_data = {
    'prompt_engineering': 'Final assignment deadline: 2026-03-15',
    'research_methods': 'Project report deadline: 2026-03-20',
}

def campus_hours(day):
    key = str(day).strip().lower()
    return campus_hours_data.get(key, 'No data for this day')

def room_capacity(room):
    key = str(room).strip().upper()
    return room_capacity_data.get(key, 'Unknown room')

def deadline_lookup(course):
    key = str(course).strip().lower()
    return deadline_data.get(key, 'Unknown course')


In [57]:
system_prompt_tools_campus = """Here are available tools:
<tools>
<tool_description>
<tool_name>campus_hours</tool_name>
<description>Returns opening hours for a specific day of week.</description>
<parameters>
<parameter><name>day</name><type>str</type></parameter>
</parameters>
</tool_description>

<tool_description>
<tool_name>room_capacity</tool_name>
<description>Returns seating capacity for a room code like A101.</description>
<parameters>
<parameter><name>room</name><type>str</type></parameter>
</parameters>
</tool_description>

<tool_description>
<tool_name>deadline_lookup</tool_name>
<description>Returns assignment deadlines by course key.</description>
<parameters>
<parameter><name>course</name><type>str</type></parameter>
</parameters>
</tool_description>
</tools>
"""

system_prompt_campus = system_prompt_tools_general + '\n' + system_prompt_tools_campus
tool_map_campus = {
    'campus_hours': campus_hours,
    'room_capacity': room_capacity,
    'deadline_lookup': deadline_lookup,
}


In [59]:
question = 'What is the seating capacity of room B202?'
print(run_tool_loop(question, system_prompt_campus, tool_map_campus, verbose=True))


<ASSISTANT>
<function_calls>
<invoke name="room_capacity">
<parameter name="room">B202</parameter>
</invoke>

<room_capacity>
40
<ASSISTANT>
The seating capacity of room B202 is 40.
The seating capacity of room B202 is 40.


In [61]:
question = 'When is the prompt engineering final assignment due?'
print(run_tool_loop(question, system_prompt_campus, tool_map_campus, verbose=True))


<ASSISTANT>
To find out when the prompt engineering final assignment is due, we need to look up the deadline for assignments related to prompt engineering. Since the exact course key is not specified, we'll assume it's a general prompt engineering course and search for deadlines based on common course structures.

<function_calls>
<invoke name="deadline_lookup">
<parameter name="course">prompt engineering</parameter>
</invoke>

<deadline_lookup>
Unknown course
<ASSISTANT>
It seems that the course "prompt engineering" is not recognized. Could you please provide more details about the course, such as its full name or the university where it is offered? This will help me locate the relevant information more accurately.
It seems that the course "prompt engineering" is not recognized. Could you please provide more details about the course, such as its full name or the university where it is offered? This will help me locate the relevant information more accurately.


### Exercise 3 (15-18 min): Make Tool Selection Reliable

Edit only `USER_PROMPT` so GigaChat calls `deadline_lookup` (not other tools).

Success criterion: output contains `<invoke name="deadline_lookup">`.


In [15]:
USER_PROMPT = '[Replace this text]'

response = get_completion(
    [{'role': 'user', 'content': USER_PROMPT}],
    system_prompt=system_prompt_campus,
    stop_sequences=stop_sequences,
)

is_correct = '<invoke name="deadline_lookup">' in response
print(response)
print('\nSolved:', is_correct)


The information requested isn't readily available without additional context. Could you please provide more details? For example, which campus or building you're referring to, or what specific type of query you're asking about (e.g., hours, capacity, deadlines)?

Solved: False


## Part 4. Real Function Calling Integration

This section adds one fully real integration:

1. **Real function calling** with a live external API (Open-Meteo).


### 4.1 Real Function Calling: Live Weather API

This tool calls real public endpoints (no API key required):
- Geocoding API: city -> coordinates
- Forecast API: coordinates -> current weather

Then GigaChat decides when to call `get_live_weather`.


In [62]:
import requests

def get_live_weather(city):
    city = str(city).strip()
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1, "language": "en", "format": "json"},
        timeout=20,
    )
    geo.raise_for_status()
    geo_data = geo.json()

    results = geo_data.get("results") or []
    if not results:
        return f"No location found for: {city}"

    place = results[0]
    lat = place["latitude"]
    lon = place["longitude"]
    resolved_name = place.get("name", city)
    country = place.get("country", "")

    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,wind_speed_10m,weather_code",
        },
        timeout=20,
    )
    weather.raise_for_status()
    current = weather.json().get("current", {})

    return (
        f"Location: {resolved_name}, {country}; "
        f"Temperature: {current.get('temperature_2m')}°C; "
        f"Wind: {current.get('wind_speed_10m')} km/h; "
        f"Weather code: {current.get('weather_code')}"
    )


In [64]:
system_prompt_tools_weather = """Here are available tools:
<tools>
<tool_description>
<tool_name>get_live_weather</tool_name>
<description>Fetches the current weather for a city using live Open-Meteo APIs.</description>
<parameters>
<parameter><name>city</name><type>str</type><description>City name in English.</description></parameter>
</parameters>
</tool_description>
</tools>
"""

system_prompt_weather = system_prompt_tools_general + "\n" + system_prompt_tools_weather

question = "What is the weather in Berlin right now? Give a short answer."
print(run_tool_loop(question, system_prompt_weather, {"get_live_weather": get_live_weather}, verbose=True))


<ASSISTANT>
<function_calls>
<invoke name="get_live_weather">
<parameter name="city">Berlin</parameter>
</invoke>

<get_live_weather>
Location: Berlin, Germany; Temperature: 6.3°C; Wind: 11.0 km/h; Weather code: 3
<ASSISTANT>
The current weather in Berlin is 6.3°C with a wind speed of 11.0 km/h and a weather code of 3.
The current weather in Berlin is 6.3°C with a wind speed of 11.0 km/h and a weather code of 3.


## Wrap-Up

You now have a practical low-code workflow for:
- function calling with GigaChat,
- tool execution loops,
- reliable tool selection through schema and prompt design.


## Part 5. GigaChain Function Agents (Priority Integration)


Learning goals:
- Design robust tool schemas with `@giga_tool` and few-shot examples.
- Build ReAct-style function agents with `create_react_agent`.
- Scale from one tool to multiple tools.
- Implement class-based tools with `BaseTool` for stricter control.
- Validate generated function schema before production use.


### 5.1 Setup for LangChain + GigaChat Function Agents

Run this setup in your environment before the next cells.


In [ ]:
# Optional: install if needed
# !pip install -U langchain-gigachat langgraph python-dotenv


In [65]:
import os
from typing import List, Optional, Literal, Type

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_gigachat.chat_models import GigaChat as LC_GigaChat
from langchain_gigachat.tools.giga_tool import giga_tool, FewShotExamples
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage
from langchain_core.tools import BaseTool
from langchain_core.callbacks import CallbackManagerForToolRun

load_dotenv()

if "GIGACHAT_CREDENTIALS" not in os.environ and "API_KEY" in os.environ:
    os.environ["GIGACHAT_CREDENTIALS"] = os.environ["API_KEY"]

if "GIGACHAT_CREDENTIALS" not in os.environ:
    raise ValueError("Set GIGACHAT_CREDENTIALS (or API_KEY) in .env before running this part.")

lc_model = LC_GigaChat(
    model="GigaChat-2-Max",
    verify_ssl_certs=False,
    streaming=False,
    max_tokens=8000,
)


/Users/daniilserki/miniconda3/envs/giga/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 5.2 Tool Design with `@giga_tool` and Few-Shot Hints

The model usually selects tools more reliably when tool descriptions and examples are explicit.


In [66]:
class SendSmsResult(BaseModel):
    status: str = Field(description="Delivery status")
    message: str = Field(description="Human-readable result")


sms_few_shots = [
    {
        "request": "Send an SMS to +1234567890 with text 'Hi, are you free today?'",
        "params": {"recipient": "+1234567890", "message": "Hi, are you free today?"},
    }
]


@giga_tool(few_shot_examples=sms_few_shots)
def send_sms(
    recipient: str = Field(description="Recipient phone number in international format"),
    message: str = Field(description="SMS body text"),
) -> SendSmsResult:
    """Send an SMS message to a recipient."""
    print(f"! send_sms -> recipient={recipient}, message={message}")
    return SendSmsResult(status="OK", message="SMS message was sent")


### 5.3 Single-Tool Agent with Memory


In [68]:
sms_tools = [send_sms]
agent_sms = create_react_agent(
    lc_model.bind_functions(sms_tools),
    sms_tools,
    checkpointer=MemorySaver(),
    # state_modifier=(
    #     "You are an SMS assistant. "
    #     "Collect missing parameters before calling send_sms."
    # ),
)

sms_response = agent_sms.invoke(
    {"messages": [HumanMessage(content="Send 'Meeting starts at 5 PM' to +15551234567")]},
    config={"configurable": {"thread_id": "sms_demo"}},
)
print(sms_response["messages"][-1].content)


/var/folders/p0/jjpv1rtx7ljfjllbr544_xqr0000gn/T/ipykernel_10937/3248176673.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_sms = create_react_agent(


! send_sms -> recipient=+15551234567, message=Meeting starts at 5 PM
The SMS message "Meeting starts at 5 PM" has been successfully sent to +15551234567.


### 5.4 Multi-Tool Agent Routing

The next two tools follow the same pattern from the priority notebook: one retrieval-like tool and one calculation-like tool.


In [73]:
class SearchMoviesResult(BaseModel):
    movies: List[str] = Field(description="Movie titles that match user filters")


@giga_tool(
    few_shot_examples=[
        {"request": "Find comedies from 2023", "params": {"genre": "comedy", "year": 2023}}
    ]
)
def search_movies(
    genre: Optional[str] = Field(default=None, description="Movie genre"),
    year: Optional[int] = Field(default=None, description="Release year"),
    actor: Optional[str] = Field(default=None, description="Actor name"),
) -> SearchMoviesResult:
    """Search movies by optional genre, year, and actor filters."""
    print(f"! search_movies -> genre={genre}, year={year}, actor={actor}")
    return SearchMoviesResult(movies=["Oppenheimer", "The Holdovers"])


class TripDistanceResult(BaseModel):
    distance_km: int = Field(description="Distance in kilometers")


@giga_tool(
    few_shot_examples=[
        {
            "request": "How far is it from Moscow to Saint Petersburg?",
            "params": {"start_location": "Moscow", "end_location": "Saint Petersburg"},
        }
    ]
)
def calculate_trip_distance(
    start_location: str = Field(description="Start city/location"),
    end_location: str = Field(description="Destination city/location"),
) -> TripDistanceResult:
    """Calculate trip distance between two locations."""
    print(f"! calculate_trip_distance -> start={start_location}, end={end_location}")
    return TripDistanceResult(distance_km=650)


In [74]:
multi_tools = [send_sms, search_movies, calculate_trip_distance]
agent_multi = create_react_agent(
    lc_model.bind_functions(multi_tools),
    multi_tools,
    checkpointer=MemorySaver(),
)

multi_response = agent_multi.invoke(
    {"messages": [HumanMessage(content="Find recent sci-fi movies and text me the best one at +15551234567")]},
    config={"configurable": {"thread_id": "multi_demo"}},
)
print(multi_response["messages"][-1].content)


/var/folders/p0/jjpv1rtx7ljfjllbr544_xqr0000gn/T/ipykernel_10937/3121121139.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_multi = create_react_agent(


! search_movies -> genre=sci-fi, year=annotation=NoneType required=False default=None description='Release year', actor=annotation=NoneType required=False default=None description='Actor name'
! send_sms -> recipient=+15551234567, message=The best recent sci-fi movie is 'Oppenheimer'.
I have found the best recent sci-fi movie for you and sent it via SMS.


### 5.5 Class-Based Tool with `BaseTool`

Use this pattern when you need strict input/output schema control or custom sync/async execution behavior.


In [71]:
class PlayerReactionsInput(BaseModel):
    reaction: Literal["add_like", "remove_like", "add_dislike"] = Field(
        description=(
            "Action to apply to currently playing content. "
            "Use add_dislike only for explicit negative feedback."
        )
    )
    content_type: Optional[Literal["track", "artist", "album", "playlist"]] = Field(
        default=None,
        description="Optional content type if user explicitly specifies it",
    )


class PlayerReactionsOutput(BaseModel):
    status: Literal["success", "fail"] = Field(description="Execution status")
    error: Optional[str] = Field(default=None, description="Error text when status=fail")


class PlayerReactionsTool(BaseTool):
    name: str = "player_reactions"
    description: str = (
        "Handle user reactions to currently playing content. "
        "After tool call, respond briefly and do not ask follow-up questions."
    )
    args_schema: Type[BaseModel] = PlayerReactionsInput
    return_schema: Type[BaseModel] = PlayerReactionsOutput
    few_shot_examples: FewShotExamples = [
        {"request": "Like this track", "params": {"reaction": "add_like", "content_type": "track"}},
        {"request": "Remove like", "params": {"reaction": "remove_like"}},
        {"request": "Dislike this artist", "params": {"reaction": "add_dislike", "content_type": "artist"}},
    ]

    def _run(
        self,
        reaction: str,
        content_type: Optional[str] = None,
        run_manager: Optional[CallbackManagerForToolRun] = None,
    ) -> PlayerReactionsOutput:
        print(f"! player_reactions -> reaction={reaction}, content_type={content_type}")
        if reaction == "add_dislike":
            return PlayerReactionsOutput(status="fail", error="Dislike is not supported in this demo backend")
        return PlayerReactionsOutput(status="success")


### 5.6 Inspect Generated Function Schema

Before deploying, inspect the schema sent to GigaChat.


In [72]:
import json
from langchain_gigachat.utils.function_calling import convert_to_gigachat_function

print(json.dumps(convert_to_gigachat_function(PlayerReactionsTool()), indent=2, ensure_ascii=False))


{
  "name": "player_reactions",
  "description": "Handle user reactions to currently playing content. After tool call, respond briefly and do not ask follow-up questions.",
  "parameters": {
    "properties": {
      "reaction": {
        "description": "Action to apply to currently playing content. Use add_dislike only for explicit negative feedback.",
        "enum": [
          "add_like",
          "remove_like",
          "add_dislike"
        ],
        "type": "string"
      },
      "content_type": {
        "default": null,
        "description": "Optional content type if user explicitly specifies it",
        "enum": [
          "track",
          "artist",
          "album",
          "playlist"
        ],
        "type": "string"
      }
    },
    "required": [
      "reaction"
    ],
    "type": "object"
  },
  "return_parameters": {
    "properties": {
      "status": {
        "description": "Execution status",
        "enum": [
          "success",
          "fail"
   

### Exercise 7: Improve Tool Selection with Few-Shot Examples

Task:
- Edit only `sms_few_shots` and movie distance few-shot blocks.
- Add 3 realistic examples with ambiguous phrasing.

Success criteria:
- The agent selects the correct tool in at least 3/4 ambiguous test prompts.


### Exercise 8: Add a New Function Tool

Task:
- Add a new `@giga_tool` named `book_taxi` with fields: `pickup`, `destination`, `time`.
- Bind it into `multi_tools` and test with one user request.

Success criteria:
- The agent calls `book_taxi` with complete arguments and returns a concise confirmation.


### Exercise 9: BaseTool for Stateful Action

Task:
- Create `ReminderStatusTool(BaseTool)` with `args_schema`, `return_schema`, and 2 few-shot examples.
- Validate its JSON schema via `convert_to_gigachat_function`.

Success criteria:
- Optional fields have defaults.
- Generated schema is valid and readable.


### Exercise 10: Unified Agent Design

Build one agent that combines:
- communication tool (`send_sms` or `book_taxi`),
- information tool (`search_movies`),
- utility tool (`calculate_trip_distance`),
- class-based tool (`PlayerReactionsTool` or your own BaseTool).

Deliverables:
- A short system prompt,
- the final tool list,
- 5 test user prompts,
- a short reflection on where tool routing failed and why.
